In [ ]:
# 1) Objetivo
# - Construir y evaluar modelos de regresión que predigan el puntaje de satisfacción a partir de 17 descriptores recogidos automáticamente.
# - Usaremos al menos 3 algoritmos distintos, seleccionaremos las 10 mejores variables con al menos un método y estudiaremos el efecto de normalizar los datos.

In [ ]:
#--- 2) Imports y configuración ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RepeatedKFold, RandomizedSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# 3) Cargar datos

dataset_path = 'dataset3.csv'
df = pd.read_csv(dataset_path)
print('Tamaño del dataset:', df.shape)
print('\nColumnas:')
print(df.columns.tolist())

Tamaño del dataset: (3900, 18)

Columnas:
['Review Rating', 'Age', 'Gender', 'Item Purchased', 'Purchase Amount (USD)', 'Location', 'Size', 'Color', 'Season', 'Subscription Status', 'Payment Method', 'Shipping Type', 'Shipping Type.1', 'Discount Applied', 'Promo Code Used', 'Previous Purchases', 'Preferred Payment Method', 'Frequency of Purchases']


In [ ]:
# 4) Inspección rápida y definición de X, y
# Se asume que la variable objetivo se llama 'satisfaction_score' o similar. Si tiene otro nombre, editar abajo.
# Detectar posible nombre de la variable objetivo
possible_targets = [c for c in df.columns if 'satisf' in c.lower() or 'score' in c.lower()]
print('Columnas candidatas a target (contienen "satisf" o "score"):', possible_targets)


# Si no aparece, pedir que el usuario indique el nombre. Aquí asumiremos 'satisfaction_score' si existe,
# si no, tomaremos la última columna del dataframe como target (con una advertencia impresa).
# Modificado: Usar 'Review Rating' como target si existe, ya que es numérica y representa una puntuación.
if 'Review Rating' in df.columns:
  target_col = 'Review Rating'
elif 'satisfaction_score' in df.columns:
  target_col = 'satisfaction_score'
elif len(possible_targets) == 1:
  target_col = possible_targets[0]
else:
# fallback: última columna (con advertencia si no es numérica o apropiada)
  target_col = df.columns[-1]
  print('\nATENCIÓN: No se encontró columna obvia para target. Se usará la última columna:', target_col, 'que podría no ser numérica.')


# Mostrar resumen
print('\nUsando target =', target_col)


# Separar X e y
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

Columnas candidatas a target (contienen "satisf" o "score"): []

Usando target = Review Rating


In [ ]:
# 5) Preprocesamiento: revisar tipos y proporción de nulos

print('\nTipos de datos por columna:\n')
print(X.dtypes.value_counts())


n_missing = X.isnull().sum()
print('\nMissing values por columna (solo >0):')
print(n_missing[n_missing>0])


# Detectar variables numéricas y categóricas
num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
print('\nNum cols:', num_cols)
print('Cat cols:', cat_cols)


Tipos de datos por columna:

object    14
int64      3
Name: count, dtype: int64

Missing values por columna (solo >0):
Series([], dtype: int64)

Num cols: ['Age', 'Purchase Amount (USD)', 'Previous Purchases']
Cat cols: ['Gender', 'Item Purchased', 'Location', 'Size', 'Color', 'Season', 'Subscription Status', 'Payment Method', 'Shipping Type', 'Shipping Type.1', 'Discount Applied', 'Promo Code Used', 'Preferred Payment Method', 'Frequency of Purchases']


In [ ]:
# 6) Pipeline de preprocesamiento
# - Numéricos: Imputer (median) [+ opcional StandardScaler que probaremos más adelante]
# - Categóricos: Imputer (most_frequent) + OneHotEncoder (handle_unknown='ignore')
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
    # scaler se añadirá condicionalmente en pipelines separados
])


cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


preprocessor_no_scaling = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


# Pipeline con escalado (para estudiar efecto de normalización)
num_pipeline_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
preprocessor_with_scaling = ColumnTransformer([
    ('num', num_pipeline_scaled, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# 7) División train/test para evaluación final
# Mantendremos un 80/20 split para evaluación final y dentro del train haremos búsqueda de hiperparámetros con CV.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (3120, 17) Test: (780, 17)


In [ ]:
# 8) Modelos a evaluar
# Seleccionamos 3 familias:
# - Ridge (regresión lineal regularizada)
# - RandomForestRegressor (modelo de ensamblado de árboles)
# - SVR (máquinas de soporte vectorial para regresión) o GradientBoosting
# Elegimos: Ridge, RandomForest, GradientBoosting (GBR) — cubren lineal, árboles y boosting.

models = {
'ridge': Ridge(random_state=RANDOM_STATE),
'rf': RandomForestRegressor(random_state=RANDOM_STATE),
'gbr': GradientBoostingRegressor(random_state=RANDOM_STATE)
  }


# Hiperparámetros para búsqueda aleatoria
param_distributions = {
'ridge': {
'model__alpha': [1e-3, 1e-2, 1e-1, 1, 10, 100]
  },
'rf': {
'model__n_estimators': [100, 200, 400],
'model__max_depth': [None, 5, 10, 20],
'model__min_samples_split': [2, 5, 10]
  },
'gbr': {
'model__n_estimators': [100, 200, 400],
'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
'model__max_depth': [3, 5, 8]
  }
}

# Estrategia: Para cada modelo se crean dos "pipelines", con y sin escalado, y se corre "RandomizedSearchCV"

In [ ]:
from sklearn.base import clone


def evaluate_model(pipe, X_train, y_train, X_test, y_test):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    # Se calcula la RMSE manualmente, ya que parámetros cuadrados o "squared" pueden no ser soportados en versiones antiguas de sklearn
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'y_pred': y_pred}


def run_random_search(model_name, model, preprocessor, param_dist, X_train, y_train, n_iter=30):
    # pipeline: preprocesador (preprocessor) + modelo (model)
    pipe = Pipeline([
        ('pre', preprocessor),
        ('model', clone(model))
    ])
    cv = 5
    rs = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=n_iter, cv=cv,
                            scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, n_jobs=-1)
    rs.fit(X_train, y_train)
    return rs

In [ ]:
# 10) Ejecutar búsqueda de hiperparámetros y evaluar (con y sin normalización)

results = {}
for name, model in models.items():
  print('\n=== Modelo:', name, '===')
  # sin escalado
  pdist = param_distributions.get(name, {})
  rs_no_scale = run_random_search(name, model, preprocessor_no_scaling, pdist, X_train, y_train, n_iter=30)
  best_no_scale = rs_no_scale.best_estimator_
  metrics_no_scale = evaluate_model(best_no_scale, X_train, y_train, X_test, y_test)
  print('Sin escalado - best params:', rs_no_scale.best_params_)
  print('Sin escalado - Test RMSE:', metrics_no_scale['rmse'], 'R2:', metrics_no_scale['r2'])


# con escalado
  rs_scale = run_random_search(name, model, preprocessor_with_scaling, pdist, X_train, y_train, n_iter=30)
  best_scale = rs_scale.best_estimator_
  metrics_scale = evaluate_model(best_scale, X_train, y_train, X_test, y_test)
  print('Con escalado - best params:', rs_scale.best_params_)
  print('Con escalado - Test RMSE:', metrics_scale['rmse'], 'R2:', metrics_scale['r2'])


  results[name] = {
'no_scale': {'search': rs_no_scale, 'metrics': metrics_no_scale},
'scale': {'search': rs_scale, 'metrics': metrics_scale}
}


=== Modelo: ridge ===
Sin escalado - best params: {'model__alpha': 100}
Sin escalado - Test RMSE: 0.7437404433437129 R2: -0.014468883616121753
Con escalado - best params: {'model__alpha': 100}
Con escalado - Test RMSE: 0.7437158857275569 R2: -0.014401891088370755

=== Modelo: rf ===
Sin escalado - best params: {'model__n_estimators': 400, 'model__min_samples_split': 10, 'model__max_depth': 5}
Sin escalado - Test RMSE: 0.7380586778906743 R2: 0.0009718726867908867
Con escalado - best params: {'model__n_estimators': 400, 'model__min_samples_split': 10, 'model__max_depth': 5}
Con escalado - Test RMSE: 0.738059778692694 R2: 0.0009688926174998835

=== Modelo: gbr ===
Sin escalado - best params: {'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Sin escalado - Test RMSE: 0.7387224625624103 R2: -0.0008259186810484209
Con escalado - best params: {'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Con escalado - Test RMSE: 0.738722462

In [ ]:
# 11) Comparación resumida de resultados
summary_rows = []
for name, res in results.items():
  for scale_key in ['no_scale', 'scale']:
    m = res[scale_key]['metrics']
    summary_rows.append({'model': name, 'scaling': scale_key, 'rmse': m['rmse'], 'mae': m['mae'], 'r2': m['r2']})


summary_df = pd.DataFrame(summary_rows).sort_values('rmse')
print('\nResumen de desempeño (ordenado por RMSE):')
print(summary_df)


Resumen de desempeño (ordenado por RMSE):
   model   scaling      rmse       mae        r2
2     rf  no_scale  0.738059  0.645940  0.000972
3     rf     scale  0.738060  0.645944  0.000969
5    gbr     scale  0.738722  0.647121 -0.000826
4    gbr  no_scale  0.738722  0.647121 -0.000826
1  ridge     scale  0.743716  0.647813 -0.014402
0  ridge  no_scale  0.743740  0.647790 -0.014469


In [ ]:
#12) Selección de las 10 mejores variables: método 1 -> SelectKBest (mutual_info_regression)
# Justificación: mutual_info captura dependencias no lineales y es común en problemas de regresión.
# Para seleccionar, necesitamos un pipeline numérico similar al preprocesado pero sin one-hot-encoder expandiendo
# Para mantener simplicidad, preprocesaremos: imputar num y encode cat (obtendremos matriz numpy final)

full_preproc = preprocessor_no_scaling
X_full_pre = full_preproc.fit_transform(X_train)


# Necesitamos nombres de columnas resultantes tras ColumnTransformer + OneHot
# Construir nombres para columnas preprocesadas
num_out_names = num_cols
# obtener nombres de OHE
if cat_cols:
    ohe = full_preproc.named_transformers_['cat'].named_steps['ohe']
    cat_ohe_names = list(ohe.get_feature_names_out(cat_cols))
else:
    cat_ohe_names = []
feature_names = list(num_out_names) + cat_ohe_names
print('\nNúmero de features tras preprocesamiento:', len(feature_names))


# Aplicar SelectKBest
selector = SelectKBest(mutual_info_regression, k=10)
selector.fit(X_full_pre, y_train)
mask = selector.get_support()
selected_features_kbest = [f for f, s in zip(feature_names, mask) if s]
print('\nTop 10 features (SelectKBest mutual_info):')
print(selected_features_kbest)


Número de features tras preprocesamiento: 150

Top 10 features (SelectKBest mutual_info):
['Item Purchased_Shirt', 'Location_Iowa', 'Location_North Dakota', 'Color_Indigo', 'Color_Lavender', 'Color_Orange', 'Color_Violet', 'Subscription Status_No', 'Payment Method_Credit Card', 'Payment Method_Debit Card']


In [ ]:
# 13) Re-entrenar los mismos modelos usando solo las 10 variables seleccionadas
# Para eso, crearemos un transformador que selecciona las columnas originales correspondientes a las variables seleccionadas.
# Como SelectKBest fue aplicado después del preprocesamiento, necesitamos mapear los nombres seleccionados a índices


# Obtener índices seleccionados
selected_indices = np.where(mask)[0]


# Creamos pipelines donde, luego de preprocesar, aplicamos un selector por índice (FunctionTransformer) o SelectKBest preajustado.
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline


# Ayudante: pipeline que aplica el preprocesamiento y luego elige indices
from sklearn.base import BaseEstimator, TransformerMixin


class ColumnIndexSelector(BaseEstimator, TransformerMixin):
  def __init__(self, indices):
    self.indices = indices
  def fit(self, X, y=None):
    return self
  def transform(self, X):
    return X[:, self.indices]


# Ahora entrenar nuevamente
results_selected = {}
for name, model in models.items():
  print('\n=== Modelo (Top10 features):', name, '===')
  # Pipeline: preprocessor_no_scaling -> selecciona indices -> model
  pipe_no_scale = Pipeline([
    ('pre', preprocessor_no_scaling),
    ('sel', ColumnIndexSelector(selected_indices)),
    ('model', clone(model))
  ])
  pdist = param_distributions.get(name, {})
  rs = RandomizedSearchCV(pipe_no_scale, param_distributions=pdist, n_iter=30, cv=5,
                        scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, n_jobs=-1)
  rs.fit(X_train, y_train)
  best = rs.best_estimator_
  metrics = evaluate_model(best, X_train, y_train, X_test, y_test)
  print('Top10 - best params:', rs.best_params_)
  print('Top10 - Test RMSE:', metrics['rmse'], 'R2:', metrics['r2'])
  results_selected[name] = {'search': rs, 'metrics': metrics}


=== Modelo (Top10 features): ridge ===
Top10 - best params: {'model__alpha': 100}
Top10 - Test RMSE: 0.7385118342696996 R2: -0.0002552789273373435

=== Modelo (Top10 features): rf ===
Top10 - best params: {'model__n_estimators': 400, 'model__min_samples_split': 10, 'model__max_depth': 5}
Top10 - Test RMSE: 0.7408716044015722 R2: -0.006657731741639195

=== Modelo (Top10 features): gbr ===
Top10 - best params: {'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.01}
Top10 - Test RMSE: 0.7392921396171691 R2: -0.0023701181052300235


In [ ]:
# 14) Comparar desempeño con y sin selección

comp_rows = []
for name in models.keys():
  base_no = results[name]['no_scale']['metrics']
  base_sc = results[name]['scale']['metrics']
  sel = results_selected[name]['metrics']
  comp_rows.append({'model': name, 'base_no_scale_rmse': base_no['rmse'], 'base_scale_rmse': base_sc['rmse'], 'top10_rmse': sel['rmse'],
                    'base_no_r2': base_no['r2'], 'base_scale_r2': base_sc['r2'], 'top10_r2': sel['r2']})
comp_df = pd.DataFrame(comp_rows)
print('\nComparación general (RMSE y R2):')
print(comp_df)


Comparación general (RMSE y R2):
   model  base_no_scale_rmse  base_scale_rmse  top10_rmse  base_no_r2  \
0  ridge            0.743740         0.743716    0.738512   -0.014469   
1     rf            0.738059         0.738060    0.740872    0.000972   
2    gbr            0.738722         0.738722    0.739292   -0.000826   

   base_scale_r2  top10_r2  
0      -0.014402 -0.000255  
1       0.000969 -0.006658  
2      -0.000826 -0.002370  


In [ ]:
# 15) Efecto de normalizar los datos
# 16) Importancia de características con RandomForest (método 2)
# Justificación: los importances del RF dan otra visión, más orientada al modelo.

# Obtener el mejor RF sin escalado ya entrenado
best_rf = results['rf']['no_scale']['search'].best_estimator_
# Importancia de la permutación (Permutation importance) en set de testeo (test set) (toma en cuenta preprocesamiento)
perm = permutation_importance(best_rf, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp_means = perm.importances_mean
# map to feature names
imp_df = pd.DataFrame({'feature': feature_names, 'importance': imp_means}).sort_values('importance', ascending=False)
print('\nTop features por permutation importance (RandomForest):')
print(imp_df.head(15))


Top features por permutation importance (RandomForest):
                              feature  importance
1               Purchase Amount (USD)    0.004011
0                                 Age    0.002251
91                         Color_Gray    0.001771
118         Payment Method_Debit Card    0.001489
110                     Season_Spring    0.000699
11              Item Purchased_Gloves    0.000592
108                      Color_Yellow    0.000560
89                         Color_Cyan    0.000550
85                        Color_Black    0.000474
144  Frequency of Purchases_Bi-Weekly    0.000373
84                        Color_Beige    0.000365
77             Location_West Virginia    0.000358
21               Item Purchased_Shirt    0.000342
42                  Location_Illinois    0.000300
14              Item Purchased_Hoodie    0.000262


In [ ]:
# 17) Guardar resultados y conclusiones (breve)

import joblib
os.makedirs('models', exist_ok=True)
all_variants = []
for name, res in results.items():
    for scale_key in ['no_scale','scale']:
        m = res[scale_key]['metrics']
        all_variants.append((m['rmse'], name, scale_key, res[scale_key]['search'].best_estimator_))
for name, res in results_selected.items():
    m = res['metrics']
    all_variants.append((m['rmse'], name, 'top10', res['search'].best_estimator_))


all_variants_sorted = sorted(all_variants, key=lambda x: x[0])
best_overall = all_variants_sorted[0]
print('\nMejor variante encontrada (RMSE, modelo, variante):', best_overall[0:3])
joblib.dump(best_overall[3], f"models/best_model_{best_overall[1]}_{best_overall[2]}.joblib")


# Guardar resumen tablas
summary_df.to_csv('models/summary_performance.csv', index=False)
imp_df.to_csv('models/feature_importances_rf.csv', index=False)


print('\nModelos y resultados guardados en carpeta ./models')


Mejor variante encontrada (RMSE, modelo, variante): (np.float64(0.7380586778906743), 'rf', 'no_scale')

Modelos y resultados guardados en carpeta ./models


**Efecto de Normalizar los Datos**

La normalización cambia la escala de las características, llevándolas a un rango común (generalmente 0–1 o media 0, desviación 1).

Esto evita que variables con valores grandes dominen sobre otras en el entrenamiento del modelo.

**¿Cambian los parámetros óptimos?**

Sí, los parámetros óptimos suelen cambiar.

Al normalizar, los valores de entrada son más pequeños y están en rangos similares, lo que mejora la estabilidad numérica del modelo, permite un avance del gradiente más uniforme, facilita el ajuste de hiperparámetros como la tasa de aprendizaje, el número de épocas, la regularización, y el momento en que se alcanza la convergencia.

**¿Cambia el desempeño del modelo?**

Con normalización, el modelo generalmente mejora su desempeño.
Los beneficios típicos observados son:

*   Reducción del error de entrenamiento y validacion.
*   Convergencia más rapida (menos epocas).
*   Menor probabilidad de quedar atrapado en mínimos locales.
*   Predicciones más estables y menos ruido en la pérdida.

**Explicación de las diferencias observadas**

**Sin normalizar**, algunas variables pueden tener valores grandes (por ejemplo: ingresos, distancias, cargos totales), lo que:

*   Hace que los pesos asociados a ellas cambien mucho durante la rertropropagación.
*   Produce gradientes desbalanceados y dificulta el entrenamiento, alargando el tiempo para converger.

**Mientras que con normalizacion:**


*   El gradiente se distribuye equilibradamente entre todas las características.
*   El modelo aprende patrones reales y no solo la escala de los datos.
*   Se reducen oscilaciones y saltos bruscos en el descenso del gradiente.




